In [0]:
from pyspark.sql.functions import col, current_timestamp

# ============ CONFIGURAÇÕES ============
catalogo = "oc_projeto"
schema = "oc-tabelas"
table_bronze = f"`{catalogo}`.`{schema}`.flights_bronze"

input_path = "/Volumes/oc_projeto/default/oc-raw/"

# IMPORTANTE: Pastas para schemaLocation e checkpointLocation
schema_location = "/Volumes/oc_projeto/default/oc-raw/_schema/"
checkpoint_location = "/Volumes/oc_projeto/default/oc-raw/_checkpoint/"
print("="*70)
print("PIPELINE AUTOLOADER - CARREGANDO NOVOS JSONs")
print("="*70)


In [0]:
# ============ PASSO 1: AUTOLOADER (CloudFiles) ============
print("\nPASSO 1: Configurando Autoloader...\n")

df_stream = (
    spark.readStream                          # ← STREAMING (não batch!)
    .format("cloudFiles")                     # ← CloudFiles monitora a pasta
    .option("cloudFiles.format", "json")      # ← Formato: JSON
    .option("cloudFiles.schemaLocation", schema_location)  # ← Onde guardar schema
    .option("cloudFiles.inferColumnTypes", "true")         # ← Inferir tipos
    .option("multiLine", "true")              # ← JSON multiline
    .load(input_path)
)

print(f"* Autoloader configurado")
print(f"* Monitorando: {input_path}")
print(f"* Schema location: {schema_location}")

In [0]:
# ============ PASSO 2: ADICIONAR METADADOS ============
print("\nPASSO 2: Adicionando metadados...\n")

df_para_bronze = df_stream.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voos_recebidos"),
    col("voos_despachados"),
    
    # Metadados (arquivo rastreado pelo Autoloader)
    col("_metadata.file_path").alias("arquivo_origem"),
    col("_metadata.file_modification_time").alias("data_modificacao_arquivo"),
    current_timestamp().alias("data_carregamento")
)

print(f"* Metadados configurados")

In [0]:
# ============ PASSO 3: ESCREVER NA BRONZE (STREAMING) ============
print("\nPASSO 3: Iniciando pipeline Autoloader...\n")

query = (
    df_para_bronze
    .writeStream
    .outputMode("append")                     
    .option("checkpointLocation", checkpoint_location)  # ← Rastreia lidos
    .trigger(availableNow=True)
    .toTable(table_bronze)
)

print(f"* Stream iniciado")
print(f"* Checkpoint location: {checkpoint_location}")
print(f"* Tabela: {table_bronze}")

In [0]:
# ============ PASSO 4: AGUARDAR PROCESSAMENTO ============
print("\nPASSO 4: Processando arquivos...\n")

# Aguarda um pouco para processar o batch
query.awaitTermination(timeout=60)

print(f"* Processamento concluído")

In [0]:
# ============ PASSO 5: VERIFICAR RESULTADO ============
print("\nPASSO 5: Verificando resultado...\n")

df_bronze = spark.table(table_bronze)
total = df_bronze.count()

print(f"* Total de registros na BRONZE: {total}")
print(f"\nÚltimos registros carregados:")

df_bronze.orderBy(col("data_carregamento").desc()).limit(3).display()

print("\n" + "="*70)
print("PIPELINE AUTOLOADER ATIVO!")
print("="*70)